# PPO Vertical-Actions Branch for Bluebird Gymnasium

This notebook is a **separate branch experiment** based on [minimal_ppo_agent.ipynb](/home/sprite/BluebirdATC/bluebird-gymnasium/examples/minimal_ppo_agent.ipynb).

Its purpose is to test the user idea of adding climb/descent actions, without contaminating
the lateral-only checkpoint lineage used by the baseline and curriculum notebooks.

Key difference from the lateral-only PPO notebooks:

- `simple_fl_climb = True`
- `simple_fl_descent = True`
- default scenario uses 2 aircraft so vertical actions have some chance of mattering

Because the action count changes, checkpoints from the lateral-only notebooks are not directly compatible.


## Environment note

Run this notebook from the same working kernel/environment as the older PPO notebooks.
It uses the same overall structure and plotting style, but starts a separate checkpoint lineage.


## Imports and path setup


In [ ]:
from __future__ import annotations

import json
import random
import shutil
import sys
from pathlib import Path

import imageio
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import Image, display

search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
gym_root = None
dt_root = None

for candidate in search_roots:
    if (candidate / 'bluebird_gymnasium').exists():
        gym_root = candidate
        sibling_dt = candidate.parent / 'bluebird-dt'
        if sibling_dt.exists():
            dt_root = sibling_dt
        break
    if (candidate / 'bluebird-gymnasium').exists() and (candidate / 'bluebird-dt').exists():
        gym_root = candidate / 'bluebird-gymnasium'
        dt_root = candidate / 'bluebird-dt'
        break

if gym_root is None or dt_root is None:
    raise RuntimeError('Could not locate local bluebird-gymnasium and bluebird-dt package roots.')

sys.path.insert(0, str(gym_root))
sys.path.insert(0, str(dt_root))

from bluebird_gymnasium.envs import EnvConfig, ViewType
from bluebird_gymnasium.envs.sector_i import SectorIEnv
from bluebird_gymnasium.envs.sector_xplus import SectorXPlusEnv

print(f'Using bluebird-gymnasium from: {gym_root}')
print(f'Using bluebird-dt from: {dt_root}')
print(f'Torch version: {torch.__version__}')


## PPO actor-critic model and agents

This is the same PPO implementation line as the older minimal PPO notebook.
The experimental change is in the environment action configuration, not the model architecture.


In [ ]:
class ActorCriticNetwork(nn.Module):
    """Shared-trunk actor-critic network for PPO."""

    def __init__(
        self,
        observation_dimension: int,
        number_of_actions: int,
        hidden_units: int = 128,
    ) -> None:
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(observation_dimension, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
        )
        self.policy_head = nn.Linear(hidden_units, number_of_actions)
        self.value_head = nn.Linear(hidden_units, 1)

    def forward(self, observation_batch: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        features = self.trunk(observation_batch)
        action_logits = self.policy_head(features)
        state_value = self.value_head(features).squeeze(-1)
        return action_logits, state_value


class PPOAgent:
    """Minimal PPO agent with actor-critic network and checkpoint helpers."""

    def __init__(
        self,
        observation_dimension: int,
        number_of_actions: int,
        learning_rate: float = 3e-4,
        hidden_units: int = 128,
        clip_epsilon: float = 0.2,
        value_loss_coefficient: float = 0.5,
        entropy_coefficient: float = 0.01,
        ppo_epochs: int = 4,
    ) -> None:
        self.observation_dimension = observation_dimension
        self.number_of_actions = number_of_actions
        self.actor_critic = ActorCriticNetwork(
            observation_dimension=observation_dimension,
            number_of_actions=number_of_actions,
            hidden_units=hidden_units,
        )
        self.optimizer = optim.Adam(self.actor_critic.parameters(), lr=learning_rate)
        self.clip_epsilon = clip_epsilon
        self.value_loss_coefficient = value_loss_coefficient
        self.entropy_coefficient = entropy_coefficient
        self.ppo_epochs = ppo_epochs

    def choose_training_action(
        self,
        observation_vector: np.ndarray,
    ) -> tuple[int, torch.Tensor, torch.Tensor, torch.Tensor]:
        observation_tensor = torch.tensor(
            observation_vector,
            dtype=torch.float32,
        ).unsqueeze(0)
        action_logits, state_value = self.actor_critic(observation_tensor)
        action_distribution = torch.distributions.Categorical(logits=action_logits)
        sampled_action = action_distribution.sample()
        action_log_probability = action_distribution.log_prob(sampled_action)

        return (
            sampled_action.item(),
            observation_tensor.squeeze(0),
            action_log_probability.squeeze(0),
            state_value.squeeze(0),
        )

    def choose_evaluation_actions(
        self,
        observation_by_callsign: dict[str, np.ndarray],
    ) -> dict[str, int]:
        chosen_actions: dict[str, int] = {}
        with torch.no_grad():
            for callsign, observation_vector in observation_by_callsign.items():
                observation_tensor = torch.tensor(
                    observation_vector,
                    dtype=torch.float32,
                ).unsqueeze(0)
                action_logits, _state_value = self.actor_critic(observation_tensor)
                chosen_actions[callsign] = torch.argmax(action_logits, dim=-1).item()
        return chosen_actions

    def update_from_trajectory(
        self,
        observations: list[torch.Tensor],
        actions: list[torch.Tensor],
        old_log_probabilities: list[torch.Tensor],
        returns: torch.Tensor,
        advantages: torch.Tensor,
    ) -> dict[str, float] | None:
        if not observations:
            return None

        observation_tensor = torch.stack(observations)
        action_tensor = torch.stack(actions).long()
        old_log_probability_tensor = torch.stack(old_log_probabilities).detach()
        returns_tensor = returns.detach()
        advantages_tensor = advantages.detach()

        if advantages_tensor.numel() > 1:
            advantages_std = advantages_tensor.std(unbiased=False)
            if advantages_std > 1e-8:
                advantages_tensor = (
                    (advantages_tensor - advantages_tensor.mean())
                    / (advantages_std + 1e-8)
                )

        mean_policy_loss = 0.0
        mean_value_loss = 0.0
        mean_entropy = 0.0
        mean_total_loss = 0.0

        for _epoch in range(self.ppo_epochs):
            new_action_logits, new_state_values = self.actor_critic(observation_tensor)
            action_distribution = torch.distributions.Categorical(logits=new_action_logits)
            new_log_probabilities = action_distribution.log_prob(action_tensor)
            entropy = action_distribution.entropy().mean()

            probability_ratio = torch.exp(new_log_probabilities - old_log_probability_tensor)
            unclipped_objective = probability_ratio * advantages_tensor
            clipped_objective = torch.clamp(
                probability_ratio,
                1.0 - self.clip_epsilon,
                1.0 + self.clip_epsilon,
            ) * advantages_tensor

            policy_loss = -torch.min(unclipped_objective, clipped_objective).mean()
            value_loss = torch.nn.functional.mse_loss(new_state_values, returns_tensor)
            total_loss = (
                policy_loss
                + self.value_loss_coefficient * value_loss
                - self.entropy_coefficient * entropy
            )

            self.optimizer.zero_grad()
            total_loss.backward()
            self.optimizer.step()

            mean_policy_loss += float(policy_loss.item())
            mean_value_loss += float(value_loss.item())
            mean_entropy += float(entropy.item())
            mean_total_loss += float(total_loss.item())

        epoch_divisor = float(self.ppo_epochs)
        return {
            'policy_loss': mean_policy_loss / epoch_divisor,
            'value_loss': mean_value_loss / epoch_divisor,
            'entropy': mean_entropy / epoch_divisor,
            'total_loss': mean_total_loss / epoch_divisor,
        }

    def save_checkpoint(self, checkpoint_path: Path, metadata: dict | None = None) -> None:
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            'model_state_dict': self.actor_critic.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'metadata': _to_python_types(metadata or {}),
        }
        torch.save(payload, checkpoint_path)

    def load_checkpoint(
        self,
        checkpoint_path: Path,
        map_location: str = 'cpu',
        load_optimizer_state: bool = True,
        strict_shape_check: bool = True,
    ) -> dict:
        payload = torch.load(checkpoint_path, map_location=map_location, weights_only=False)
        metadata = payload.get('metadata', {})
        if strict_shape_check:
            saved_obs_dim = metadata.get('observation_dimension')
            saved_num_actions = metadata.get('number_of_actions')
            if saved_obs_dim is not None and saved_obs_dim != self.observation_dimension:
                raise ValueError(
                    f'Checkpoint observation dimension {saved_obs_dim} does not match current {self.observation_dimension}. '
                    'Use a different checkpoint lineage.'
                )
            if saved_num_actions is not None and saved_num_actions != self.number_of_actions:
                raise ValueError(
                    f'Checkpoint action count {saved_num_actions} does not match current {self.number_of_actions}. '
                    'Use a different checkpoint lineage.'
                )
        self.actor_critic.load_state_dict(payload['model_state_dict'])
        if load_optimizer_state and 'optimizer_state_dict' in payload:
            self.optimizer.load_state_dict(payload['optimizer_state_dict'])
        return metadata



def _to_python_types(value):
    if isinstance(value, dict):
        return {key: _to_python_types(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [_to_python_types(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    return value


class RandomAgent:
    """Simple random baseline for comparison."""

    def __init__(self, number_of_actions: int) -> None:
        self.number_of_actions = number_of_actions

    def choose_evaluation_actions(
        self,
        observation_by_callsign: dict[str, np.ndarray],
    ) -> dict[str, int]:
        return {
            callsign: random.randrange(self.number_of_actions)
            for callsign in observation_by_callsign.keys()
        }


## Environment configuration and rollout helpers

This branch keeps the decentralized PPO structure, but turns on vertical actions.
It also defaults to 2 aircraft so the extra actions are not completely idle.


In [ ]:
def make_training_config(
    env_cls,
    num_aircraft: int,
    k_nearest_aircraft: int = 1,
    enable_vertical_actions: bool = False,
) -> EnvConfig:
    config = env_cls.get_default_env_config(ViewType.DECENTRALIZED)

    config.state_repr_config = {
        'encoder_cls': 'extra_minimal',
        'k_nearest_aircraft': k_nearest_aircraft,
    }

    config.action_config = {
        'simple_heading_left': True,
        'simple_heading_right': True,
        'simple_fl_climb': enable_vertical_actions,
        'simple_fl_descent': enable_vertical_actions,
        'simple_fl_exit': False,
    }

    config.reward_config = {
        'fns': [
            'position_status_const',
            'lateral_centreline_distance_shaped',
            'safety_simple_avoidance_exp',
        ],
        'coeffs': [1.0, 1.0, 1.2],
    }

    config.view_config = {
        'type': ViewType.DECENTRALIZED.value,
        'decentralized_params': {},
    }

    config.scenario_config = {
        'cls': 'tactical',
        'args': {
            'num_aircraft': num_aircraft,
            'balance': [0.0, 0.0, 1.0],
        },
    }

    return config


def compute_returns_and_advantages(
    rewards: list[float],
    values: list[torch.Tensor],
    dones: list[bool],
    discount_factor_gamma: float,
    gae_lambda: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    rewards_tensor = torch.tensor(rewards, dtype=torch.float32)
    values_tensor = torch.stack(values).detach().float()
    dones_tensor = torch.tensor(dones, dtype=torch.float32)

    advantages = torch.zeros_like(rewards_tensor)
    last_gae = torch.tensor(0.0)

    for timestep in reversed(range(len(rewards))):
        if timestep == len(rewards) - 1:
            next_value = torch.tensor(0.0)
        else:
            next_value = values_tensor[timestep + 1]

        next_nonterminal = 1.0 - dones_tensor[timestep]
        delta = (
            rewards_tensor[timestep]
            + discount_factor_gamma * next_value * next_nonterminal
            - values_tensor[timestep]
        )
        last_gae = delta + discount_factor_gamma * gae_lambda * next_nonterminal * last_gae
        advantages[timestep] = last_gae

    returns = advantages + values_tensor
    return returns, advantages


def run_one_training_episode(
    environment,
    agent: PPOAgent,
    random_seed: int,
    discount_factor_gamma: float,
    gae_lambda: float,
) -> tuple[float, int, dict[str, float] | None]:
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    observation_by_callsign, _info = environment.reset(seed=random_seed)

    episode_is_done = False
    episode_step_count = 0
    episode_total_reward = 0.0

    observations: list[torch.Tensor] = []
    actions: list[torch.Tensor] = []
    old_log_probabilities: list[torch.Tensor] = []
    values: list[torch.Tensor] = []
    rewards: list[float] = []
    dones: list[bool] = []

    while not episode_is_done:
        action_by_callsign: dict[str, int] = {}
        actions_this_step = 0

        for callsign, observation_vector in observation_by_callsign.items():
            action_int, observation_tensor, action_log_probability, state_value = agent.choose_training_action(observation_vector)
            action_by_callsign[callsign] = action_int
            observations.append(observation_tensor)
            actions.append(torch.tensor(action_int))
            old_log_probabilities.append(action_log_probability.detach())
            values.append(state_value.detach())
            actions_this_step += 1

        (
            next_observation_by_callsign,
            reward_by_callsign,
            done_by_callsign,
            truncated_by_callsign,
            _info,
        ) = environment.step(action_by_callsign)

        timestep_reward = float(sum(reward_by_callsign.values())) if reward_by_callsign else 0.0
        timestep_done = all(done_by_callsign.values()) if done_by_callsign else True

        rewards.extend([timestep_reward] * max(actions_this_step, 1))
        dones.extend([bool(timestep_done)] * max(actions_this_step, 1))

        _ = truncated_by_callsign
        episode_total_reward += timestep_reward
        episode_is_done = timestep_done
        observation_by_callsign = next_observation_by_callsign
        episode_step_count += 1

    returns, advantages = compute_returns_and_advantages(
        rewards=rewards,
        values=values,
        dones=dones,
        discount_factor_gamma=discount_factor_gamma,
        gae_lambda=gae_lambda,
    )

    update_metrics = agent.update_from_trajectory(
        observations=observations,
        actions=actions,
        old_log_probabilities=old_log_probabilities,
        returns=returns,
        advantages=advantages,
    )

    return episode_total_reward, episode_step_count, update_metrics


def run_one_evaluation_episode(environment, evaluation_agent, random_seed: int) -> tuple[float, int]:
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    observation_by_callsign, _info = environment.reset(seed=random_seed)

    episode_is_done = False
    episode_step_count = 0
    episode_total_reward = 0.0

    while not episode_is_done:
        action_by_callsign = evaluation_agent.choose_evaluation_actions(observation_by_callsign)
        (
            next_observation_by_callsign,
            reward_by_callsign,
            done_by_callsign,
            truncated_by_callsign,
            _info,
        ) = environment.step(action_by_callsign)

        _ = truncated_by_callsign
        episode_total_reward += float(sum(reward_by_callsign.values())) if reward_by_callsign else 0.0
        episode_is_done = all(done_by_callsign.values()) if done_by_callsign else True
        observation_by_callsign = next_observation_by_callsign
        episode_step_count += 1

    return episode_total_reward, episode_step_count


def evaluate_agent_over_seeds(environment, evaluation_agent, evaluation_seeds: list[int]) -> dict:
    rewards: list[float] = []
    steps: list[int] = []

    for random_seed in evaluation_seeds:
        total_reward, step_count = run_one_evaluation_episode(environment, evaluation_agent, random_seed)
        rewards.append(total_reward)
        steps.append(step_count)

    return {
        'seeds': evaluation_seeds,
        'rewards': rewards,
        'steps': steps,
        'mean_reward': float(np.mean(rewards)),
        'std_reward': float(np.std(rewards)),
        'mean_steps': float(np.mean(steps)),
    }


def render_evaluation_rollout_to_gif(
    env_cls,
    stage_num_aircraft: int,
    enable_vertical_actions: bool,
    agent: PPOAgent,
    random_seed: int,
    render_dir: Path,
    gif_name: str,
    render_every_n_steps: int = 5,
    gif_frame_duration_seconds: float = 0.2,
) -> Path:
    render_config = make_training_config(
        env_cls=env_cls,
        num_aircraft=stage_num_aircraft,
        enable_vertical_actions=enable_vertical_actions,
    )
    render_config.radar_config['display_actions'] = True
    render_config.radar_config['render_dir'] = str(render_dir)
    render_config.radar_config['prefix'] = 'frame'

    if render_dir.exists():
        shutil.rmtree(render_dir)
    render_dir.mkdir(parents=True, exist_ok=True)

    render_environment = env_cls(config=render_config)
    render_environment.set_render_mode('file')

    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    observation_by_callsign, _info = render_environment.reset(seed=random_seed)
    render_environment.render()

    episode_is_done = False
    step_index = 0

    while not episode_is_done:
        action_by_callsign = agent.choose_evaluation_actions(observation_by_callsign)
        (
            next_observation_by_callsign,
            reward_by_callsign,
            done_by_callsign,
            truncated_by_callsign,
            _info,
        ) = render_environment.step(action_by_callsign)
        _ = reward_by_callsign, truncated_by_callsign
        step_index += 1
        episode_is_done = all(done_by_callsign.values()) if done_by_callsign else True
        if step_index % render_every_n_steps == 0 or episode_is_done:
            render_environment.render()
        observation_by_callsign = next_observation_by_callsign

    png_frames = sorted(render_dir.glob(f"{render_config.radar_config['prefix']}_*.png"))
    if not png_frames:
        raise RuntimeError(
            f'No rendered PNG frames were written to {render_dir}. '
            'Expected at least one frame before GIF generation.'
        )

    gif_path = render_dir / f'{gif_name}.gif'
    images = [imageio.v3.imread(frame_path) for frame_path in png_frames]
    imageio.mimsave(gif_path, images, loop=0, duration=gif_frame_duration_seconds)
    render_environment.close()
    return gif_path


## Set up the environment and inspect the shapes

This is where the branch diverges from the baseline.
You should expect a larger `number_of_actions` than the lateral-only notebooks.


In [ ]:
config = make_training_config(
    env_cls=SectorIEnv,
    num_aircraft=2,
    enable_vertical_actions=True,
)
environment = SectorIEnv(config=config)

observation_dimension = environment.observation_space.shape[0]
number_of_actions = environment.action_space.n

print(
    'environment shapes:',
    f'observation_dimension={observation_dimension}',
    f'number_of_actions={number_of_actions}',
)


## Hyperparameters and experiment settings

This branch is more exploratory, so the defaults stay conservative:

- same network width as the baseline
- same PPO structure
- separate checkpoint directory
- no attempt to load checkpoints from the lateral-only notebooks


In [ ]:
learning_rate = 3e-4
hidden_units = 128
discount_factor_gamma = 0.99
gae_lambda = 0.95
clip_epsilon = 0.2
value_loss_coefficient = 0.5
entropy_coefficient = 0.01
ppo_epochs = 4
number_of_training_episodes = 100
training_seed_start = 500
periodic_eval_interval = 10
heldout_evaluation_seeds = list(range(600, 610))
render_every_n_steps = 5
gif_frame_duration_seconds = 0.2

checkpoint_dir = Path.cwd() / 'checkpoints' / 'ppo_vertical_actions_agent'
latest_checkpoint_path = checkpoint_dir / 'latest.pt'
best_checkpoint_path = checkpoint_dir / 'best.pt'
resume_from_checkpoint: Path | None = None
load_optimizer_state = True

agent = PPOAgent(
    observation_dimension=observation_dimension,
    number_of_actions=number_of_actions,
    learning_rate=learning_rate,
    hidden_units=hidden_units,
    clip_epsilon=clip_epsilon,
    value_loss_coefficient=value_loss_coefficient,
    entropy_coefficient=entropy_coefficient,
    ppo_epochs=ppo_epochs,
)
random_agent = RandomAgent(number_of_actions=number_of_actions)

if resume_from_checkpoint is not None:
    loaded_metadata = agent.load_checkpoint(
        resume_from_checkpoint,
        load_optimizer_state=load_optimizer_state,
        strict_shape_check=True,
    )
    print('Resumed from checkpoint:', resume_from_checkpoint)
    print('Loaded metadata:', loaded_metadata)

training_rewards: list[float] = []
training_steps: list[int] = []
training_policy_losses: list[float] = []
training_value_losses: list[float] = []
training_entropies: list[float] = []
training_total_losses: list[float] = []

periodic_eval_episodes: list[int] = []
periodic_eval_learned_mean_rewards: list[float] = []
periodic_eval_learned_std_rewards: list[float] = []
periodic_eval_random_mean_rewards: list[float] = []
periodic_eval_random_std_rewards: list[float] = []

best_mean_evaluation_reward = float('-inf')


## PPO training loop with periodic evaluation and checkpointing


In [ ]:
for episode_index in range(number_of_training_episodes):
    random_seed = training_seed_start + episode_index
    total_reward, step_count, update_metrics = run_one_training_episode(
        environment=environment,
        agent=agent,
        random_seed=random_seed,
        discount_factor_gamma=discount_factor_gamma,
        gae_lambda=gae_lambda,
    )

    training_rewards.append(total_reward)
    training_steps.append(step_count)
    training_policy_losses.append(float('nan') if update_metrics is None else update_metrics['policy_loss'])
    training_value_losses.append(float('nan') if update_metrics is None else update_metrics['value_loss'])
    training_entropies.append(float('nan') if update_metrics is None else update_metrics['entropy'])
    training_total_losses.append(float('nan') if update_metrics is None else update_metrics['total_loss'])

    print(
        '[train]',
        f'episode={episode_index:03d}',
        f'seed={random_seed}',
        f'reward={total_reward:.3f}',
        f'steps={step_count}',
        f'policy_loss={None if update_metrics is None else update_metrics["policy_loss"]:.6f}' if update_metrics is not None else 'policy_loss=None',
        f'value_loss={None if update_metrics is None else update_metrics["value_loss"]:.6f}' if update_metrics is not None else 'value_loss=None',
    )

    should_run_periodic_eval = (
        (episode_index + 1) % periodic_eval_interval == 0
        or episode_index == number_of_training_episodes - 1
    )

    if should_run_periodic_eval:
        learned_eval = evaluate_agent_over_seeds(
            environment=environment,
            evaluation_agent=agent,
            evaluation_seeds=heldout_evaluation_seeds,
        )
        random_eval = evaluate_agent_over_seeds(
            environment=environment,
            evaluation_agent=random_agent,
            evaluation_seeds=heldout_evaluation_seeds,
        )

        periodic_eval_episodes.append(episode_index + 1)
        periodic_eval_learned_mean_rewards.append(learned_eval['mean_reward'])
        periodic_eval_learned_std_rewards.append(learned_eval['std_reward'])
        periodic_eval_random_mean_rewards.append(random_eval['mean_reward'])
        periodic_eval_random_std_rewards.append(random_eval['std_reward'])

        metadata = {
            'episode': episode_index + 1,
            'train_seed': random_seed,
            'environment': 'SectorIEnv',
            'view_type': 'decentralized',
            'num_aircraft': 2,
            'observation_dimension': observation_dimension,
            'number_of_actions': number_of_actions,
            'enable_vertical_actions': True,
            'learned_mean_reward': learned_eval['mean_reward'],
            'learned_std_reward': learned_eval['std_reward'],
            'random_mean_reward': random_eval['mean_reward'],
            'random_std_reward': random_eval['std_reward'],
            'evaluation_seeds': heldout_evaluation_seeds,
        }
        agent.save_checkpoint(latest_checkpoint_path, metadata=metadata)

        if learned_eval['mean_reward'] > best_mean_evaluation_reward:
            best_mean_evaluation_reward = learned_eval['mean_reward']
            agent.save_checkpoint(best_checkpoint_path, metadata=metadata)
            checkpoint_note = 'new best checkpoint'
        else:
            checkpoint_note = 'latest checkpoint only'

        print(
            '[periodic-eval]',
            f'episode={episode_index + 1:03d}',
            f'learned_mean_reward={learned_eval["mean_reward"]:.3f}',
            f'random_mean_reward={random_eval["mean_reward"]:.3f}',
            checkpoint_note,
        )


## Restore the best evaluated model


In [ ]:
if not best_checkpoint_path.exists():
    raise FileNotFoundError(f'Best checkpoint not found: {best_checkpoint_path}')

loaded_metadata = agent.load_checkpoint(
    best_checkpoint_path,
    load_optimizer_state=False,
    strict_shape_check=True,
)
print('Reloaded best checkpoint from:', best_checkpoint_path)
print('Best checkpoint metadata:')
loaded_metadata


## Final evaluation of the best checkpoint vs random baseline


In [ ]:
best_policy_eval = evaluate_agent_over_seeds(
    environment=environment,
    evaluation_agent=agent,
    evaluation_seeds=heldout_evaluation_seeds,
)
random_policy_eval = evaluate_agent_over_seeds(
    environment=environment,
    evaluation_agent=random_agent,
    evaluation_seeds=heldout_evaluation_seeds,
)

print('Best PPO policy evaluation mean reward:', best_policy_eval['mean_reward'])
print('Best PPO policy evaluation std reward:', best_policy_eval['std_reward'])
print('Random baseline mean reward:', random_policy_eval['mean_reward'])
print('Random baseline std reward:', random_policy_eval['std_reward'])


## Useful plots

This keeps the same plot style as the older PPO line so comparisons stay easy.


In [ ]:
def moving_average(values: list[float], window: int) -> np.ndarray:
    if len(values) < window:
        return np.array([])
    kernel = np.ones(window) / window
    return np.convolve(np.asarray(values, dtype=float), kernel, mode='valid')

plot_window = min(10, len(training_rewards))
smoothed_rewards = moving_average(training_rewards, plot_window)
training_episode_indices = np.arange(1, len(training_rewards) + 1)
heldout_seed_indices = np.arange(len(heldout_evaluation_seeds))
periodic_eval_episodes_arr = np.asarray(periodic_eval_episodes)
learned_mean_arr = np.asarray(periodic_eval_learned_mean_rewards)
learned_std_arr = np.asarray(periodic_eval_learned_std_rewards)
random_mean_arr = np.asarray(periodic_eval_random_mean_rewards)
random_std_arr = np.asarray(periodic_eval_random_std_rewards)

fig, axes = plt.subplots(3, 2, figsize=(15, 14))

axes[0, 0].plot(training_episode_indices, training_rewards, marker='o', alpha=0.25, label='raw reward')
if len(smoothed_rewards) > 0:
    axes[0, 0].plot(
        np.arange(plot_window, len(training_rewards) + 1),
        smoothed_rewards,
        linewidth=2.5,
        color='tab:blue',
        label=f'moving average (window={plot_window})',
    )
axes[0, 0].set_title('Training Reward per Episode')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Total Reward')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(training_episode_indices, training_steps, marker='o', color='tab:orange')
axes[0, 1].set_title('Training Episode Length')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Steps')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(periodic_eval_episodes_arr, learned_mean_arr, marker='o', label='learned policy')
axes[1, 0].fill_between(
    periodic_eval_episodes_arr,
    learned_mean_arr - learned_std_arr,
    learned_mean_arr + learned_std_arr,
    alpha=0.2,
)
axes[1, 0].plot(periodic_eval_episodes_arr, random_mean_arr, marker='s', label='random baseline')
axes[1, 0].fill_between(
    periodic_eval_episodes_arr,
    random_mean_arr - random_std_arr,
    random_mean_arr + random_std_arr,
    alpha=0.2,
)
axes[1, 0].set_title('Periodic Evaluation: Learned vs Random')
axes[1, 0].set_xlabel('Training Episode')
axes[1, 0].set_ylabel('Mean Evaluation Reward')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(
    heldout_seed_indices,
    best_policy_eval['rewards'],
    marker='o',
    linewidth=2,
    label='best PPO checkpoint',
)
axes[1, 1].plot(
    heldout_seed_indices,
    random_policy_eval['rewards'],
    marker='s',
    linewidth=2,
    label='random baseline',
)
axes[1, 1].set_xticks(heldout_seed_indices)
axes[1, 1].set_xticklabels([str(seed) for seed in heldout_evaluation_seeds], rotation=45)
axes[1, 1].set_title('Final Evaluation Reward by Seed')
axes[1, 1].set_xlabel('Held-out Evaluation Seed')
axes[1, 1].set_ylabel('Total Reward')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

axes[2, 0].plot(training_episode_indices, training_policy_losses, label='policy loss')
axes[2, 0].plot(training_episode_indices, training_value_losses, label='value loss')
axes[2, 0].plot(training_episode_indices, training_total_losses, label='total loss')
axes[2, 0].set_title('PPO Loss Terms')
axes[2, 0].set_xlabel('Episode')
axes[2, 0].set_ylabel('Loss')
axes[2, 0].legend()
axes[2, 0].grid(alpha=0.3)

axes[2, 1].plot(training_episode_indices, training_entropies, label='entropy', color='tab:green')
axes[2, 1].set_title('Policy Entropy During Training')
axes[2, 1].set_xlabel('Episode')
axes[2, 1].set_ylabel('Entropy')
axes[2, 1].grid(alpha=0.3)

fig.suptitle('PPO Vertical-Actions Branch Summary', fontsize=16)
fig.tight_layout()
plt.show()


## Render the best checkpoint and save a GIF


In [ ]:
gif_seed = heldout_evaluation_seeds[0]
render_dir = Path.cwd() / 'renders' / 'ppo_vertical_actions_agent_eval'
gif_path = render_evaluation_rollout_to_gif(
    env_cls=SectorIEnv,
    stage_num_aircraft=2,
    enable_vertical_actions=True,
    agent=agent,
    random_seed=gif_seed,
    render_dir=render_dir,
    gif_name=f'best_vertical_branch_eval_seed_{gif_seed}',
    render_every_n_steps=render_every_n_steps,
    gif_frame_duration_seconds=gif_frame_duration_seconds,
)

print(f'Saved GIF to: {gif_path}')
display(Image(filename=str(gif_path)))


## Optional cleanup


In [ ]:
environment.close()
